# Reviewed ResNet50 retraining
This separate run preserves all historical files. It verifies all 400 raw drawings, excludes conflicting duplicate groups without changing human labels, checks the 363 candidate PNGs, then trains a fresh ImageNet-initialized baseline for ten epochs. Outputs go to new run folders. This is research, not clinical validation. Participant independence and source provenance remain unconfirmed.

In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth, io, json, hashlib, uuid, time, platform, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
credentials, _ = google.auth.default()
api = build('drive','v3',credentials=credentials,cache_discovery=False)
RUN = 'resnet50_v2_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
ROOT = Path('/content')/RUN
ROOT.mkdir()
RAW=ROOT/'raw'; RAW.mkdir()
IMAGES=ROOT/'processed'; IMAGES.mkdir()
OUT=ROOT/'outputs';OUT.mkdir()
LABELS=['Closed_Circle','Number_correct_Position','Includes_all_12_correct_Numbers','Places_Hands_in_correct_positions']
NAMES=['Closed Circle','Correct Number Position','All 12 Numbers','Correct Hands']
def children(fid):
    answer=[];token=None
    while True:
        r=api.files().list(q=f"'{fid}' in parents and trashed = false",fields='nextPageToken,files(id,name,size,md5Checksum)',pageSize=1000,pageToken=token).execute(num_retries=3)
        answer.extend(r.get('files',[]));token=r.get('nextPageToken')
        if not token:return answer
def read_bytes(fid):return api.files().get_media(fileId=fid).execute(num_retries=3)
original_bytes=read_bytes('1MAzqBC_0KUjUcK1Wy7GgH4IbTKe_6yM6')
candidate_bytes=read_bytes('1x5GOtslIxTnUhksNq6aJQTq1SSSY8bbT')
original=pd.read_csv(io.BytesIO(original_bytes));df=pd.read_csv(io.BytesIO(candidate_bytes))
assert len(original)==400 and original.Image_ID.is_unique
assert len(df)==363 and df.Image_ID.is_unique
assert df['split'].value_counts().to_dict()=={'train':254,'test':55,'validation':54}
assert df[LABELS].isin([0,1]).all().all()
assert (df[LABELS].sum(axis=1)==df.Human_Total).all()
old=original.set_index('Image_ID'); new=df.set_index('Image_ID')
assert old.loc[new.index,LABELS+['Human_Total']].equals(new[LABELS+['Human_Total']])
(OUT/'dataset_split_v2.csv').write_bytes(candidate_bytes)
print('RUN:',RUN,'Candidate split:',df['split'].value_counts().to_dict(),flush=True)


In [ ]:
raw_files=[f for f in children('1xsYt8aAtHzObI4eBMhMxklqgfPEVQp6e') if Path(f['name']).suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}]
processed_files=[f for f in children('1j2F9IAD2jn4mhxrFMC5e2XWEsOXyFTo0') if f['name'].endswith('.png')]
assert len(raw_files)==400 and {Path(f['name']).stem for f in raw_files}==set(original.Image_ID)
assert len(processed_files)==400 and len({f['name'] for f in processed_files})==400
jobs=[(f,RAW) for f in raw_files]+[(f,IMAGES) for f in processed_files if Path(f['name']).stem in set(df.Image_ID)]
def download(job):
    f,folder=job
    service=build('drive','v3',credentials=credentials,cache_discovery=False)
    data=service.files().get_media(fileId=f['id']).execute(num_retries=3)
    if f.get('md5Checksum'):assert hashlib.md5(data).hexdigest()==f['md5Checksum']
    p=folder/f['name'];p.write_bytes(data)
    return p
with ThreadPoolExecutor(max_workers=6) as pool:
    for n,p in enumerate(pool.map(download,jobs),1):
        if n%50==0 or n==len(jobs):print('Downloaded and verified',n,'/',len(jobs),flush=True)
pixel_groups=defaultdict(list); raw_hashes=[]
for p in sorted(RAW.iterdir()):
    with Image.open(p) as image:
        image.load();image=ImageOps.exif_transpose(image).convert('RGBA')
        rgb=Image.alpha_composite(Image.new('RGBA',image.size,(255,255,255,255)),image).convert('RGB')
    digest=hashlib.sha256(str(rgb.size).encode()+rgb.tobytes()).hexdigest()
    pixel_groups[digest].append(p.stem)
    raw_hashes.append({'Image_ID':p.stem,'file_sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'pixel_sha256':digest})
eligible=[];conflicts=[];duplicates=[]
for g in pixel_groups.values():
    g=sorted(g)
    if len(g)>1:duplicates.append(g)
    if len({tuple(old.loc[i,LABELS]) for i in g})>1:conflicts.append(g)
    else:eligible.append(g[0])
assert set(eligible)==set(df.Image_ID),'Fresh pixel audit differs from candidate. STOP and review; no training.'
processed_hashes=[]
for row in df.itertuples():
    p=IMAGES/(row.Image_ID+'.png')
    with Image.open(p) as image:
        image.load();assert image.mode=='RGB' and image.size==(224,224)
        digest=hashlib.sha256(image.tobytes()).hexdigest()
    processed_hashes.append({'Image_ID':row.Image_ID,'split':row.split,'pixel_sha256':digest})
ph=pd.DataFrame(processed_hashes)
assert ph.pixel_sha256.is_unique,'Duplicate processed pixels remain. STOP before training.'
audit={'run_id':RUN,'raw_records':400,'candidate_records':len(df),'raw_duplicate_groups':duplicates,'conflicting_groups':conflicts,'candidate_processed_pixel_duplicates':0,'participant_independence':'unverified','source_manifest_sha256':hashlib.sha256(original_bytes).hexdigest(),'candidate_manifest_sha256':hashlib.sha256(candidate_bytes).hexdigest()}
(OUT/'data_validation.json').write_text(json.dumps(audit,indent=2))
pd.DataFrame(raw_hashes).to_csv(OUT/'raw_image_hashes.csv',index=False)
ph.to_csv(OUT/'processed_image_hashes.csv',index=False)
print('PASS: fresh raw audit matches candidate; 363 unique processed pixel arrays; no exact content overlap.',flush=True)
print('Conflicting groups held out:',len(conflicts),'records:',sum(map(len,conflicts)),flush=True)


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(42)
for gpu in tf.config.list_physical_devices('GPU'):tf.config.experimental.set_memory_growth(gpu,True)
def load_split(name):
    part=df[df['split']==name].sort_values('Image_ID')
    X=np.stack([np.asarray(Image.open(IMAGES/(i+'.png')),dtype=np.float32) for i in part.Image_ID])
    return part,X,part[LABELS].to_numpy(dtype=np.float32)
train,X_train,y_train=load_split('train');valid,X_val,y_val=load_split('validation')
base=ResNet50(weights='imagenet',include_top=False,input_shape=(224,224,3));base.trainable=False
inputs=layers.Input((224,224,3),name='clock_image')
x=preprocess_input(inputs);x=base(x,training=False);x=layers.GlobalAveragePooling2D()(x);x=layers.Dropout(.30)(x)
model=tf.keras.Model(inputs,layers.Dense(4,activation='sigmoid',name='criteria_predictions')(x))
model.compile(optimizer=tf.keras.optimizers.Adam(.001),loss='binary_crossentropy',metrics=[tf.keras.metrics.BinaryAccuracy(name='binary_accuracy')])
print('TensorFlow',tf.__version__,'GPUs:',tf.config.list_physical_devices('GPU'),flush=True)
start=time.time()
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=16,verbose=2,callbacks=[tf.keras.callbacks.CSVLogger(str(OUT/'training_history_live.csv'))])
elapsed=time.time()-start
hist=pd.DataFrame(history.history);hist.insert(0,'epoch',range(1,len(hist)+1));hist.to_csv(OUT/'training_history.csv',index=False)
model.save(OUT/'clock_drawing_resnet50_v2.keras')
print('TRAINING COMPLETE:',len(hist),'epochs',round(elapsed,1),'seconds',flush=True)


In [ ]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix
test,X_test,y_test=load_split('test')
prob=model.predict(X_test,batch_size=16,verbose=0);pred=(prob>=.5).astype(int)
metrics=[]
for j,name in enumerate(NAMES):
    tn,fp,fn,tp=confusion_matrix(y_test[:,j],pred[:,j],labels=[0,1]).ravel()
    metrics.append({'Criterion':name,'N':len(test),'Accuracy':accuracy_score(y_test[:,j],pred[:,j]),'Precision':precision_score(y_test[:,j],pred[:,j],zero_division=0),'Recall':recall_score(y_test[:,j],pred[:,j],zero_division=0),'F1':f1_score(y_test[:,j],pred[:,j],zero_division=0),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)})
metrics=pd.DataFrame(metrics);metrics.to_csv(OUT/'baseline_metrics.csv',index=False)
prediction=test[['Image_ID','Human_Total']].copy();prediction['AI_Total']=pred.sum(axis=1)
for j,c in enumerate(LABELS):prediction[c+'_human']=y_test[:,j].astype(int);prediction[c+'_probability']=prob[:,j];prediction[c+'_predicted']=pred[:,j]
prediction.to_csv(OUT/'test_predictions.csv',index=False)
err=prediction.AI_Total-prediction.Human_Total
summary={'run_id':RUN,'status':'trained_and_evaluated','images':363,'train':254,'validation':54,'test':55,'epochs':10,'batch_size':16,'seed':42,'threshold':.5,'exact_score_matches':int((err==0).sum()),'exact_score_agreement':float((err==0).mean()),'score_mae':float(abs(err).mean()),'within_one_point':int((abs(err)<=1).sum()),'exact_four_label_matches':int(np.all(y_test==pred,axis=1).sum()),'binary_accuracy':float((y_test==pred).mean()),'training_seconds':elapsed,'tensorflow':tf.__version__,'python':platform.python_version(),'gpu':[str(g) for g in tf.config.list_physical_devices('GPU')],'final_training':{k:float(v[-1]) for k,v in history.history.items()},'limitations':['Conflicting labels excluded, not adjudicated','Exact duplicate control does not establish participant independence or exclude near duplicates','Retrospective revised split; not an external validation cohort','Metrics not directly comparable to historical run on a different sample']}
(OUT/'run_summary.json').write_text(json.dumps(summary,indent=2))
for metric in ['loss','binary_accuracy']:
    plt.figure(figsize=(7,4));plt.plot(hist.epoch,hist[metric],label='Train');plt.plot(hist.epoch,hist['val_'+metric],label='Validation');plt.xlabel('Epoch');plt.ylabel(metric.replace('_',' '));plt.legend();plt.tight_layout();plt.savefig(OUT/('baseline_'+metric+'.png'),dpi=160);plt.close()
counts=pd.DataFrame({'Fail':[(df[c]==0).sum() for c in LABELS],'Pass':[(df[c]==1).sum() for c in LABELS]},index=NAMES)
counts.to_csv(OUT/'criteria_distribution.csv');counts.plot.bar(figsize=(8,4));plt.xticks(rotation=20,ha='right');plt.tight_layout();plt.savefig(OUT/'criteria_distribution.png',dpi=160);plt.close()
score_counts=df.Human_Total.value_counts().reindex(range(5),fill_value=0).sort_index();score_counts.rename_axis('Human_Total').to_csv(OUT/'total_score_distribution.csv');score_counts.plot.bar();plt.xlabel('Human score');plt.ylabel('Records');plt.tight_layout();plt.savefig(OUT/'total_score_distribution.png',dpi=160);plt.close()
reloaded=tf.keras.models.load_model(OUT/'clock_drawing_resnet50_v2.keras')
assert np.allclose(prob,reloaded.predict(X_test,batch_size=16,verbose=0),atol=1e-6),'Reload verification failed'
print(metrics.to_string(index=False));print(json.dumps(summary,indent=2),flush=True)


In [ ]:
def new_folder(name,parent):
    return api.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder','parents':[parent]},fields='id,webViewLink').execute(num_retries=3)
results_folder=new_folder(RUN,'1aEE3ms9AHj1D-Ix0r0aJFwqOUrHW0KSC')
model_folder=new_folder(RUN,'1aW6wjYMkbEK_vtP1WHqUvDobZ4t9Oa5O')
saved=[]
for path in sorted(OUT.iterdir()):
    parent=model_folder['id'] if path.suffix=='.keras' else results_folder['id']
    media=MediaFileUpload(str(path),resumable=True)
    request=api.files().create(body={'name':path.name,'parents':[parent]},media_body=media,fields='id,name,md5Checksum,webViewLink')
    response=None
    while response is None:_,response=request.next_chunk(num_retries=3)
    assert response['md5Checksum']==hashlib.md5(path.read_bytes()).hexdigest()
    saved.append(response);print('Uploaded and verified:',path.name,flush=True)
print('SUCCESS: Retraining, evaluation, model reload and all Drive uploads verified.',flush=True)
print('RESULTS_FOLDER:',results_folder)
print('MODEL_FOLDER:',model_folder)
print('FILES:',json.dumps(saved),flush=True)


In [ ]:
# Package only the reviewed preprocessed dataset; never include raw drawings.
import zipfile
package_path=ROOT/'Preprocessed_Dataset_v2_RESTRICTED.zip'
readme='RESTRICTED EDUCATIONAL RESEARCH DATA. 363 RGB 224x224 PNGs, split 254/54/55. No raw drawings included. Do not upload this archive or its images to GitHub or public documentation. NHATS source release, license and scoring protocol remain unconfirmed; verify permitted recipient access before redistribution. Labels unchanged; conflicting duplicate groups excluded, not adjudicated. Use packaged_image_path from package_manifest.csv. Original path columns retain source lineage. dataset_version=v2_candidate is retained exactly from the evaluated manifest; completed run: '+RUN
pack=df.copy()
pack['packaged_image_path']=['processed_images/'+s+'/'+i+'.png' for s,i in zip(pack['split'],pack.Image_ID)]
with zipfile.ZipFile(package_path,'w',zipfile.ZIP_DEFLATED) as z:
    z.writestr('README_RESTRICTED.txt',readme)
    z.writestr('package_manifest.csv',pack.to_csv(index=False))
    for r in pack.itertuples():z.write(IMAGES/(r.Image_ID+'.png'),r.packaged_image_path)
    for n in ['dataset_split_v2.csv','processed_image_hashes.csv','data_validation.json']:z.write(OUT/n,n)
with zipfile.ZipFile(package_path) as z:
    assert z.testzip() is None
    assert sum(n.endswith('.png') for n in z.namelist())==363
    for r in pack.itertuples():
        with Image.open(io.BytesIO(z.read(r.packaged_image_path))) as im:
            assert hashlib.sha256(im.tobytes()).hexdigest()==ph.set_index('Image_ID').loc[r.Image_ID,'pixel_sha256']
folder=new_folder('Preprocessed_Dataset_v2_RESTRICTED','1NfuKHBf5pjbAGW9hojLJOqhDzqt9Nx78')
r=api.files().create(body={'name':package_path.name,'parents':[folder['id']]},media_body=MediaFileUpload(str(package_path),resumable=True),fields='id,name,md5Checksum,webViewLink').execute(num_retries=3)
assert r['md5Checksum']==hashlib.md5(package_path.read_bytes()).hexdigest()
permissions=api.permissions().list(fileId=r['id'],fields='permissions(type,role)').execute()['permissions']
assert not any(p['type'] in ('anyone','domain') for p in permissions)
print('DATASET PACKAGE VERIFIED: 363 images; all hashes match; no public/domain access.',r,folder)
